In [1]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import xarray as xr

In [2]:
lake = 'zug'
folder_path = r"/storage/alplakes_test/" + lake + "_100m_2025"
input_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_final")

output_folder = os.path.join(folder_path, "outputs_swirl", "ke_eddy")
os.makedirs(output_folder, exist_ok=True)

In [3]:
lvl0_csv_path = os.path.join(input_folder, "lvl0.csv")
lake_csv_path = os.path.join(input_folder, "lake_characteristics.csv")

# Get datasets

In [4]:
df_lvl0 = pd.read_csv(lvl0_csv_path)
df_lvl0 = df_lvl0.set_index('id', drop=False)
df_lvl0['date'] = pd.to_datetime(df_lvl0['date'])

In [5]:
df_lvl0['i_eddy_cells'][0]

'28.0,28.0,28.0,28.0,29.0,29.0,29.0,29.0,29.0,29.0,29.0,30.0,30.0,30.0,30.0,30.0,30.0,30.0,30.0,31.0,31.0,31.0,31.0,31.0,31.0,31.0,31.0,31.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,33.0,33.0,33.0,33.0,33.0,33.0,33.0,33.0,33.0,34.0,34.0,34.0,34.0,34.0,34.0,34.0,34.0,34.0,35.0,35.0,35.0,35.0,35.0,35.0,35.0,35.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,37.0,37.0,37.0,37.0,37.0,37.0,37.0,37.0,37.0,37.0,37.0,38.0,38.0,38.0,38.0,38.0,38.0,38.0,38.0,38.0,38.0,39.0,39.0,39.0,39.0,39.0,39.0,39.0,39.0,39.0,40.0,40.0,40.0,41.0,41.0,41.0,42.0'

# Get eddy indices

In [6]:
def parse_cell_string(s):
    if pd.isna(s) or s == "":
        return np.array([], dtype=float)
    return np.fromstring(s, sep=",")

In [7]:
grouped = (
    df_lvl0.groupby(["time_index", "depth_index"], sort=False)
      .agg(
          i_eddy_cells_concat=("i_eddy_cells", lambda s: ",".join(s.dropna().astype(str))),
          j_eddy_cells_concat=("j_eddy_cells", lambda s: ",".join(s.dropna().astype(str))),
      )
      .reset_index()
)

In [8]:
grouped["i_eddy_cells_array"] = grouped["i_eddy_cells_concat"].apply(parse_cell_string)
grouped["j_eddy_cells_array"] = grouped["j_eddy_cells_concat"].apply(parse_cell_string)

In [9]:
grouped['itime_index_corrected'] = grouped['time_index'] - 1

In [10]:
np.concat(grouped["i_eddy_cells_array"]).max()

np.float64(46.0)

# Get KE dataset

In [11]:
ds_ke = xr.open_dataarray(r"/storage/alplakes_test/" + lake + "_100m_2025/energy_budget/kinetic_energy.nc")

# Create mask DataArray

In [12]:
mask = xr.DataArray(
    np.zeros(ds_ke.shape, dtype=bool),
    coords=ds_ke.coords,
    dims=ds_ke.dims,
)

In [13]:
for _, row in grouped.iterrows():
    t = int(row["itime_index_corrected"])
    z = int(row["depth_index"])

    ii = np.asarray(row["i_eddy_cells_array"], dtype=int)   # or row["i_eddy_cells_array"]
    jj = np.asarray(row["j_eddy_cells_array"], dtype=int)   # or row["j_eddy_cells_array"]

    # skip empty
    if len(ii) == 0:
        continue

    mask.values[t, z, jj, ii] = True


IndexError: index 8760 is out of bounds for axis 0 with size 8760

In [ ]:
mask.name = "eddy_mask"

encoding = {
    "eddy_mask": {
        "zlib": True,
        "complevel": 4,     # 1–9, 4 is a good default
        "dtype": "i1"       # store as int8 instead of bool (smaller / more compatible)
    }
}

mask.astype("i1").to_netcdf("/storage/alplakes_test/"+lake+"_100m_2025/outputs_swirl/eddy_catalogues_final/eddy_mask.nc", encoding=encoding)
